# 03 — Model Development & Training

**HemoVisionAI** — Acute Lymphoblastic Leukemia (ALL) Classification

Trains **HemoVisionNet V1** using the stratified data pipeline.
All configuration from `config.py`; data loading from `preprocessing.data_pipeline`.

**Architecture:** Input → Augmentation → Rescaling(1/255) → Stem(32) → Feature(64) → Feature(128) → Feature(256) → GAP → Dense(256) → Dropout(0.5) → Softmax(4)

---

In [2]:
# ============================================================
# Cell 1: Imports and Project Setup
# ============================================================
import os
import sys
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Central configuration — single source of truth
from config import (
    DATASET_PATH, IMAGE_SIZE, BATCH_SIZE, SEED, VAL_SPLIT,
    NUM_CLASSES, LEARNING_RATE, EPOCHS,
    CHECKPOINT_DIR, CHECKPOINT_PATH,
)

# Data pipeline — single implementation
from preprocessing.data_pipeline import get_datasets

print(f"TensorFlow version: {tf.__version__}")
print(f"Checkpoint path:    {CHECKPOINT_PATH}")

I0000 00:00:1785808206.162447   14716 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785808206.164109   14716 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785808206.204292   14716 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785808207.516192   14716 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

TensorFlow version: 2.21.0
Checkpoint path:    /home/chirag/ML_project1/HemoVisionAI/saved_models/hemovisionnet_v1_best.keras


## 1. Load Data

In [3]:
# ============================================================
# Cell 2: Load Datasets via Shared Pipeline
# ============================================================
# get_datasets() handles: directory scan → DataFrame → stratified split →
# print distribution tables → tf.data.Dataset with [0, 255] float32 pixels.

train_ds, val_ds, class_names = get_datasets(
    dataset_dir=DATASET_PATH,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    seed=SEED,
    augment_fn=None,  # augmentation is inside the model
)

# Verify pixel range
for images, _ in train_ds.take(1):
    assert tf.reduce_max(images).numpy() > 1.0, (
        "Pipeline delivers [0,1] — model has Rescaling(1/255) and will double-normalise!"
    )
print("✅ Pixel range verified: [0, 255]")

✅ Built DataFrame: 3242 images across 4 classes
✅ Stratified split: 2593 train / 649 val

  TRAINING SET — Class Distribution
Class                             Count   Percentage
----------------------------------------------------
Benign                              409      15.77 %
[Malignant] Pre-B                   764      29.46 %
[Malignant] Pro-B                   637      24.57 %
[Malignant] early Pre-B             783      30.20 %
----------------------------------------------------
TOTAL                              2593     100.00 %

  VALIDATION SET — Class Distribution
Class                             Count   Percentage
----------------------------------------------------
Benign                              103      15.87 %
[Malignant] Pre-B                   191      29.43 %
[Malignant] Pro-B                   159      24.50 %
[Malignant] early Pre-B             196      30.20 %
----------------------------------------------------
TOTAL                               649 

E0000 00:00:1785808211.670907   14716 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✅ Pixel range verified: [0, 255]


## 2. Data Augmentation

In [4]:
# ============================================================
# Cell 3: Augmentation (applied inside model — training only)
# ============================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="DataAugmentation")

## 3. Model Architecture — HemoVisionNet V1

In [5]:
# ============================================================
# Cell 4: Building Blocks
# ============================================================
def hemo_stem_block(inputs):
    """Initial conv block for low-level feature extraction."""
    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    return x

def hemo_feature_block(inputs, filters):
    """Double-conv block for hierarchical feature extraction."""
    x = layers.Conv2D(filters, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    return x

In [6]:
# ============================================================
# Cell 5: Assemble HemoVisionNet V1
# ============================================================
inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="Input_Image")

x = data_augmentation(inputs)

# Normalise [0, 255] → [0, 1] INSIDE the model.
# The data pipeline delivers raw [0, 255] — DO NOT also normalise there.
x = layers.Rescaling(1.0 / 255, name="Normalization")(x)

x = hemo_stem_block(x)
x = hemo_feature_block(x, 64)
x = hemo_feature_block(x, 128)
x = hemo_feature_block(x, 256)

x = layers.GlobalAveragePooling2D(name="GlobalAveragePooling")(x)
x = layers.Dense(256, activation="relu", name="Dense_256")(x)
x = layers.Dropout(0.5, name="Dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="Prediction")(x)

hemovisionnet = Model(inputs=inputs, outputs=outputs, name="HemoVisionNet_V1")
hemovisionnet.summary()

assert hemovisionnet.output_shape[-1] == NUM_CLASSES

Model: "HemoVisionNet_V1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_Image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ DataAugmentation (Sequential)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Normalization (Rescaling)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 1,233,540 (4.71 MB)

 Trainable params: 1,231,684 (4.70 MB)

 Non-trainable params: 1,856 (7.25 KB)

## 4. Compile & Callbacks

In [7]:
# ============================================================
# Cell 6: Compile
# ============================================================
hemovisionnet.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [8]:
# ============================================================
# Cell 7: Callbacks
# ============================================================
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True, verbose=1,
)
model_checkpoint = ModelCheckpoint(
    CHECKPOINT_PATH, monitor="val_accuracy", save_best_only=True, verbose=1,
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=4, min_lr=1e-6, verbose=1,
)

callbacks = [early_stopping, model_checkpoint, reduce_lr]
print(f"✅ Checkpoint will be saved to: {CHECKPOINT_PATH}")

✅ Checkpoint will be saved to: /home/chirag/ML_project1/HemoVisionAI/saved_models/hemovisionnet_v1_best.keras


## 5. Train

In [9]:
# ============================================================
# Cell 8: Train
# ============================================================
history = hemovisionnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)
print("\n✅ Training complete!")

Epoch 1/50


/home/chirag/ML_project1/HemoVisionAI/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


: 

## 6. Training Curves

In [ ]:
# ============================================================
# Cell 9: Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"],     label="Train", linewidth=2)
axes[0].plot(history.history["val_accuracy"],  label="Val",   linewidth=2)
axes[0].set_title("Accuracy");  axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["loss"],     label="Train", linewidth=2)
axes[1].plot(history.history["val_loss"],  label="Val",   linewidth=2)
axes[1].set_title("Loss");  axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("HemoVisionNet V1 — Training History", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()